# ANLP Assignment 3 - Part 1 & Part 2 Dual-GPU Kaggle Runner
This notebook automates repository setup, dataset linking from `/kaggle/input`, dual-GPU training & evaluation for Part 1, and Part 2 GNN experiments.

In [ ]:
import os
from pathlib import Path

# Clone repository if not already cloned
if not Path("anlp_assign_3").exists() and not Path("src").exists():
    !git clone https://github.com/AnishRacherla/anlp_assign_3.git
    %cd anlp_assign_3
elif Path("anlp_assign_3").exists() and not Path("src").exists():
    %cd anlp_assign_3

# Pull latest code and scripts
!git pull origin main

## 1. Setup Environment Variables (Credentials & Hyperparameters)
*Using `os.environ` in Python ensures variables persist across all cells in Kaggle.*

In [ ]:
import os
from pathlib import Path

# Optional: Paste your API keys here if you want WandB and Hugging Face uploading:
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")
os.environ["WANDB_API_KEY"] = os.environ.get("WANDB_API_KEY", "")

# Auto-load from .env if present
if Path(".env").exists():
    with open(".env") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip().strip("'\"")

# Target Repository and WandB Project
os.environ.setdefault("ROLL_NUMBER", "2024101046")
os.environ.setdefault("HF_REPO_ID", "AnishRacherla/anlp-a3-models")
os.environ.setdefault("WANDB_PROJECT", "anlp-a3-part1")
os.environ.setdefault("PYTHONUNBUFFERED", "1")

# Training Configuration (Full 570k dataset with 3 epochs takes ~70 min across dual GPUs)
os.environ["EPOCHS"] = "3"
os.environ["BATCH_SIZE"] = "32"
os.environ["MAX_SEQ_LEN"] = "512"
# Set to "" to train on all 569,670 sequences (recommended). Set to "25000" only for quick dry-runs.
os.environ["MAX_TRAIN_SAMPLES"] = ""

print("✓ Environment variables configured successfully!")
print(f"HF_REPO_ID: {os.environ.get('HF_REPO_ID')}")
print(f"WANDB_PROJECT: {os.environ.get('WANDB_PROJECT')}")
print(f"MAX_TRAIN_SAMPLES: {os.environ.get('MAX_TRAIN_SAMPLES') or 'FULL DATASET (569,670 sequences)'}")

## 2. Locate & Link Dataset from `/kaggle/input`
*Recursively scans for `cipher.jsonl` anywhere inside `/kaggle/input` so nesting structure never causes issues.*

In [ ]:
from pathlib import Path
import shutil

print("Scanning /kaggle/input for cipher dataset...")
cipher_files = list(Path("/kaggle/input").rglob("cipher.jsonl"))

if not cipher_files:
    raise FileNotFoundError("Could not find any 'cipher.jsonl' in /kaggle/input! Please verify dataset is attached.")

print(f"Found {len(cipher_files)} cipher file(s):")
for cf in cipher_files:
    print(" -", cf)

Path("data/part1").mkdir(parents=True, exist_ok=True)

# Copy train, validation, and test splits
for split in ["train", "validation", "test"]:
    src_dir = None
    for cf in cipher_files:
        if split in cf.parts:
            src_dir = cf.parent
            break
            
    if src_dir and src_dir.exists():
        dst_dir = Path("data/part1") / split
        if dst_dir.exists():
            shutil.rmtree(dst_dir)
        shutil.copytree(src_dir, dst_dir)
        files = [p.name for p in dst_dir.glob("*")]
        print(f"Copied {split} -> {dst_dir} ({len(files)} files: {files})")
    else:
        print(f"Warning: Could not locate '{split}' split in /kaggle/input")

print("\nVerifying data/part1/train:")
!ls -lh data/part1/train/

## 3. Run Part 1: Parallel Training for Hybrid & DeltaNet
- **GPU 0**: Hybrid (Mamba-DeltaNet)
- **GPU 1**: DeltaNet

In [ ]:
!bash scripts/part1/run_kaggle.sh

## 4. Live Progress Monitor
*Run this cell while training is running in the background to watch progress bars update in real time.*

In [ ]:
import time
from IPython.display import clear_output

print("Monitoring progress (updates every 10s). Stop anytime with the interrupt button.")
try:
    for _ in range(60):
        clear_output(wait=True)
        print("=== LIVE PROGRESS: HYBRID (GPU 0) & DELTANET (GPU 1) ===\n")
        print("--- GPU 0: Hybrid ---")
        !tail -n 8 logs/part1/gpu0_hybrid.log 2>/dev/null || true
        print("\n--- GPU 1: DeltaNet ---")
        !tail -n 8 logs/part1/gpu1_deltanet.log 2>/dev/null || true
        time.sleep(10)
except KeyboardInterrupt:
    print("\nMonitoring paused.")

## 5. View Part 1 Metrics & Output Artifacts

In [ ]:
import json
from pathlib import Path

print("=== PART 1 EVALUATION RESULTS ===")
metrics_files = sorted(Path("results/part1").glob("*.json"))
if not metrics_files:
    print("No metric files found in results/part1 yet.")
for p in metrics_files:
    print(f"\n--- {p.name} ---")
    print(json.dumps(json.loads(p.read_text()), indent=2))

print("\n=== GENERATED PREDICTION ARTIFACTS ===")
!ls -lh output/part1/

## 6. Run Part 2: GNN Benchmark & Feature Ablations (Optional)

In [ ]:
!pip install -q torch_geometric
!python -m src.gnn_pipeline --device cuda --push-to-hf

## 7. View Part 2 Results

In [ ]:
import json
from pathlib import Path

p2_file = Path("output/part2/results.json")
if p2_file.exists():
    print(json.dumps(json.loads(p2_file.read_text()), indent=2))
else:
    print("Part 2 results not found yet.")